# NAFNet H2 Kaggle Submission Notebook (Balanced)
This notebook:
1. Runs LR noisy -> HR restored inference on `/kaggle/input/competition-data/test/NoisyLR`
2. Saves `.npy` predictions to `/kaggle/working/submission`
3. Writes `/kaggle/working/submission.csv` with base64-encoded `.npy` payloads

In [ ]:
# Optional dependency install (uncomment if needed)
# !pip install -q pyyaml pandas tqdm

import os
import sys

# Ensure your repository is available to notebook runtime.
# Example: if code is attached as Kaggle Dataset and mounted at /kaggle/input/nafnet-h2-code
# sys.path.append('/kaggle/input/nafnet-h2-code')

In [ ]:
# Run inference (single-process works fine on Kaggle).
# Point train_output_dir to dataset path containing best_infer.pt / best.pt etc.
!python infer_nafnet_kaggle_best_infer.py \
  --train-output-dir /kaggle/input/nafnet-h2-checkpoints \
  --quality-preset balanced \
  --input-dir /kaggle/input/competition-data/test/NoisyLR \
  --output-dir /kaggle/working/submission \
  --batch-size 4 \
  --expected-size 256 \
  --verify-all-inputs

In [ ]:
# Required submission.csv generation format
import os
import numpy as np
import base64
import pandas as pd
from io import BytesIO

submission_dir = '/kaggle/working/submission'

rows = []
files = sorted([f for f in os.listdir(submission_dir) if f.endswith('.npy')])

for idx, file in enumerate(files, start=1):
    path = os.path.join(submission_dir, file)
    arr = np.load(path)

    # Validation: 256x256 float32 with finite values
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    assert arr.shape == (256, 256), f'Bad shape for {file}: {arr.shape}'
    arr = arr.astype(np.float32, copy=False)
    assert np.all(np.isfinite(arr)), f'NaN/Inf found in {file}'

    buffer = BytesIO()
    np.save(buffer, arr)
    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({'id': idx, 'npy_base64': encoded})

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission created with', len(df), 'rows')
print('Saved to /kaggle/working/submission.csv')